# ABYSSAL AI — Fine-Tuning de Segmentación Submarina (`YOLO11-seg` + `TrashCan 1.0`)

**Proyecto Final de Inteligencia Artificial — Línea 2: Visión Artificial (Segmentación en Video)**  
**Equipo:** Samuel · Fabián · Giraldo

> **Requisito en Google Colab:** Antes de ejecutar, ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: GPU (T4)**.

### ¿Qué resuelve este Notebook?
1. Descarga el dataset real de vehículos submarinos **TrashCan 1.0** (JAMSTEC / Universidad de Minnesota, 7,212 imágenes RGB reales).
2. Convierte sus máscaras poligonales COCO a formato **YOLO-seg** consolidado en **4 clases operativas para ROVs**:
   - `0`: `debris_flexible_polymer` *(bolsas, botellas, redes, cuerdas, plásticos, textiles)*
   - `1`: `debris_rigid_other` *(latas, metales, contenedores, tuberías, restos rígidos)*
   - `2`: `protected_marine_life` *(peces, cangrejos, estrellas de mar, corales, plantas)*
   - `3`: `rov_hardware` *(componentes visibles del brazo o cuerpo del ROV)*
3. Ejecuta *fine-tuning* sobre **`yolo11n-seg.pt`** en GPU T4, valida `mAP@50` / `mAP@50-95` (Box y Mask) y exporta a **ONNX**.
4. Pre-renderiza los videos sincronizados (`rov_raw.mp4` y `rov_segmented.mp4`) + `telemetry_frames.json` + `metrics_summary.json` dentro de `abyssal_web_bundle.zip` para actualizar la presentación web estilo *Awwwards*.


## 1. Configuración del entorno y verificación de GPU


In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib tqdm

import csv
import json
import math
import os
import shutil
import subprocess
import time
import zipfile
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import ultralytics
import yaml

ultralytics.checks()
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo activo: {torch.cuda.get_device_name(0)}")


## 2. Descarga oficial de `TrashCan 1.0` (JAMSTEC / UMN)

Descargamos directamente el archivo oficial `dataset.zip` desde el repositorio DRUM de la Universidad de Minnesota y localizamos los splits COCO (`instances_train_trashcan.json` e `instances_val_trashcan.json`).


In [ ]:
RAW_DIR = Path("/content/trashcan_raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = RAW_DIR / "dataset.zip"

TRASHCAN_URL = "https://conservancy.umn.edu/server/api/core/bitstreams/fffa78cc-3abf-42b9-b019-bc15a953a8e9/content"

if ZIP_PATH.exists() and not zipfile.is_zipfile(ZIP_PATH):
    print("Eliminando archivo previo corrupto o inválido...")
    ZIP_PATH.unlink()

if not ZIP_PATH.exists():
    !curl -L -f --retry 3 --progress-bar -A "Mozilla/5.0" -o /content/trashcan_raw/dataset.zip "{TRASHCAN_URL}"

assert zipfile.is_zipfile(ZIP_PATH), f"El archivo descargado no es un ZIP válido: {ZIP_PATH}"

if not (RAW_DIR / "extracted_ok.flag").exists():
    print("Descomprimiendo dataset/instance_version/ desde dataset.zip...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        members = [m for m in zf.namelist() if m.startswith("dataset/instance_version/")]
        if not members:
            members = zf.namelist()
        zf.extractall(RAW_DIR, members=members)
    (RAW_DIR / "extracted_ok.flag").write_text("ok")

VERSION_DIR = RAW_DIR / "dataset" / "instance_version"
if VERSION_DIR.is_dir():
    TRAIN_JSON = VERSION_DIR / "instances_train_trashcan.json"
    VAL_JSON = VERSION_DIR / "instances_val_trashcan.json"
else:
    train_jsons = sorted(RAW_DIR.rglob("*instances_train_trashcan.json"))
    val_jsons = sorted(RAW_DIR.rglob("*instances_val_trashcan.json"))
    assert train_jsons and val_jsons, "No se encontraron los JSON COCO de TrashCan tras descomprimir."
    TRAIN_JSON = next((p for p in train_jsons if "instance_version" in str(p)), train_jsons[0])
    VAL_JSON = next((p for p in val_jsons if "instance_version" in str(p)), val_jsons[0])

VERSION_ROOT = TRAIN_JSON.parent

def find_split_images_dir(root: Path, split_name: str) -> Path:
    for cand in [root / split_name, root / "images" / split_name, *root.rglob(split_name)]:
        if cand.is_dir() and any(cand.glob("*.jpg")):
            return cand
    raise FileNotFoundError(f"No se encontraron imágenes para '{split_name}' en {root}")

TRAIN_IMG_SRC = find_split_images_dir(VERSION_ROOT, "train")
VAL_IMG_SRC = find_split_images_dir(VERSION_ROOT, "val")

print("TRAIN_JSON:", TRAIN_JSON)
print("VAL_JSON:  ", VAL_JSON)
print("TRAIN_IMGS:", TRAIN_IMG_SRC)
print("VAL_IMGS:  ", VAL_IMG_SRC)


## 3. Conversión COCO → YOLO-seg, Taxonomía Operativa y EDA

Consolidamos las clases originales de `TrashCan` en **4 clases operativas** enfocadas en decisiones de manipulación robótica segura, validamos que cada polígono normalizado esté dentro de `[0, 1]`, generamos `data.yaml` y visualizamos la distribución de clases + ejemplos reales con sus máscaras.


In [ ]:
YOLO_DIR = Path("/content/trashcan_yolo")
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

OPERATIONAL_CLASSES = {
    0: "debris_flexible_polymer",
    1: "debris_rigid_other",
    2: "protected_marine_life",
    3: "rov_hardware",
}
FLEXIBLE_KEYWORDS = (
    "plastic", "bag", "bottle", "net", "rope", "wrapper",
    "cup", "tarp", "rubber", "fabric", "cloth", "fishing", "foam"
)
BIO_NAMES = {"bio", "coral", "fish", "crab", "starfish", "eel", "shells"}

def map_trashcan_category(cat_name: str):
    name = cat_name.strip().lower()
    if not name:
        return None
    if name == "rov" or name.startswith("rov"):
        return 3
    if name.startswith("animal") or name.startswith("plant") or name in BIO_NAMES:
        return 2
    if any(k in name for k in FLEXIBLE_KEYWORDS):
        return 0
    if name.startswith("trash") or name.startswith("debris"):
        return 1
    return None

def normalize_polygon(coords, width, height):
    if width <= 0 or height <= 0 or len(coords) < 6 or len(coords) % 2 != 0:
        return None
    norm, xs, ys = [], [], []
    for i in range(0, len(coords), 2):
        nx = min(1.0, max(0.0, float(coords[i]) / width))
        ny = min(1.0, max(0.0, float(coords[i + 1]) / height))
        norm.extend((nx, ny))
        xs.append(nx)
        ys.append(ny)
    if (max(xs) - min(xs)) <= 1e-6 or (max(ys) - min(ys)) <= 1e-6:
        return None
    return norm

def convert_split(coco_json: Path, src_imgs: Path, split: str):
    out_imgs = YOLO_DIR / "images" / split
    out_lbls = YOLO_DIR / "labels" / split
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_lbls.mkdir(parents=True, exist_ok=True)

    coco = json.loads(coco_json.read_text(encoding="utf-8"))
    cat_map = {}
    for cat in coco.get("categories", []):
        mapped = map_trashcan_category(cat["name"])
        if mapped is not None:
            cat_map[int(cat["id"])] = mapped

    anns_by_img = {}
    for ann in coco.get("annotations", []):
        anns_by_img.setdefault(int(ann["image_id"]), []).append(ann)

    counts = Counter()
    written = 0
    for img in coco.get("images", []):
        fname = Path(img["file_name"]).name
        src_file = src_imgs / fname
        if not src_file.is_file():
            continue
        w, h = int(img.get("width") or 0), int(img.get("height") or 0)
        if w <= 0 or h <= 0:
            im = cv2.imread(str(src_file))
            if im is None:
                continue
            h, w = im.shape[:2]

        lines = []
        for ann in anns_by_img.get(int(img["id"]), []):
            cid = int(ann.get("category_id", -1))
            if cid not in cat_map:
                continue
            seg = ann.get("segmentation")
            if not isinstance(seg, list):
                continue
            rings = [seg] if all(isinstance(v, (int, float)) for v in seg) else [r for r in seg if isinstance(r, list)]
            t_cls = cat_map[cid]
            for ring in rings:
                norm = normalize_polygon(ring, w, h)
                if norm is not None:
                    lines.append(f"{t_cls} " + " ".join(f"{v:.6f}" for v in norm))
                    counts[t_cls] += 1

        dst_img = out_imgs / src_file.name
        if not dst_img.exists():
            os.symlink(src_file.resolve(), dst_img)
        (out_lbls / f"{src_file.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        written += 1

    return written, counts

n_train, c_train = convert_split(TRAIN_JSON, TRAIN_IMG_SRC, "train")
n_val, c_val = convert_split(VAL_JSON, VAL_IMG_SRC, "val")

data_yaml = {
    "path": str(YOLO_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": 4,
    "names": OPERATIONAL_CLASSES,
}
(YOLO_DIR / "data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")

print(f"Train imágenes: {n_train} | Polígonos por clase: {dict(sorted(c_train.items()))}")
print(f"Val   imágenes: {n_val} | Polígonos por clase: {dict(sorted(c_val.items()))}")

# EDA Visual: distribución y muestras con ground-truth
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
labels_names = [OPERATIONAL_CLASSES[i] for i in range(4)]
train_vals = [c_train[i] for i in range(4)]
val_vals = [c_val[i] for i in range(4)]
x = np.arange(4)
axes[0].bar(x - 0.18, train_vals, width=0.35, label="Train", color="#00F0FF")
axes[0].bar(x + 0.18, val_vals, width=0.35, label="Val", color="#00FF9D")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_names, rotation=18, ha="right")
axes[0].set_title("Distribución de Instancias (Taxonomía Operativa)")
axes[0].legend()

# Muestra visual con polígonos
sample_lbls = [p for p in sorted((YOLO_DIR / "labels" / "train").glob("*.txt")) if p.read_text().strip()][:1]
if sample_lbls:
    lbl_p = sample_lbls[0]
    img_p = next((YOLO_DIR / "images" / "train").glob(f"{lbl_p.stem}.*"))
    vis = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    vh, vw = vis.shape[:2]
    colors_rgb = {0: (0, 240, 255), 1: (255, 183, 3), 2: (0, 255, 157), 3: (139, 92, 246)}
    for line in lbl_p.read_text().splitlines():
        parts = line.split()
        cid = int(parts[0])
        pts = np.array([[float(parts[i]) * vw, float(parts[i + 1]) * vh] for i in range(1, len(parts), 2)], np.int32)
        cv2.polylines(vis, [pts], True, colors_rgb.get(cid, (255, 255, 255)), 2)
    axes[1].imshow(vis)
    axes[1].set_title(f"Muestra Ground Truth: {img_p.name}")
    axes[1].axis("off")

plt.tight_layout()
plt.show()


## 4. Fine-Tuning de `YOLO11n-seg` en GPU T4

Ajustamos el modelo preentrenado en COCO (`yolo11n-seg.pt`) con aumentos de color pensados para variaciones submarinas (turbidez, balance azul/verde, reflexión especular).


In [ ]:
from ultralytics import YOLO

RUNS_PROJECT = "/content/runs_abyssal"
RUN_NAME = "yolo11n_seg_trashcan"

model = YOLO("yolo11n-seg.pt")
train_results = model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    cos_lr=True,
    hsv_h=0.02,
    hsv_s=0.6,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    project=RUNS_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0 if torch.cuda.is_available() else "cpu",
)

RUN_DIR = Path(RUNS_PROJECT) / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("Pesos óptimos guardados en:", BEST_WEIGHTS)


## 5. Evaluación, Exportación ONNX y Generación de `metrics_summary.json`

Validamos el mejor checkpoint sobre `val`, exportamos a ONNX para despliegue ligero en dispositivos embebidos (edge ROV) y construimos el JSON real que alimenta las gráficas de la presentación web.


In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
val_metrics = best_model.val(data=str(YOLO_DIR / "data.yaml"), split="val", imgsz=640)
onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True)
print("Modelo exportado a ONNX:", onnx_path)

# Parseo de results.csv -> metrics_summary.json
def safe_float(row, keys, default=0.0):
    for k in keys:
        if k in row and row[k] != "":
            try:
                return float(row[k])
            except ValueError:
                pass
    return default

epochs_data = []
with (RUN_DIR / "results.csv").open("r", encoding="utf-8") as fh:
    for idx, raw_row in enumerate(csv.DictReader(fh), start=1):
        row = {k.strip(): (v.strip() if v else "") for k, v in raw_row.items() if k}
        epochs_data.append({
            "epoch": int(round(safe_float(row, ("epoch",), idx))),
            "train_box_loss": round(safe_float(row, ("train/box_loss",)), 4),
            "train_seg_loss": round(safe_float(row, ("train/seg_loss",)), 4),
            "train_cls_loss": round(safe_float(row, ("train/cls_loss",)), 4),
            "val_seg_loss": round(safe_float(row, ("val/seg_loss",)), 4),
            "precision_mask": round(safe_float(row, ("metrics/precision(M)", "metrics/precision(B)")), 4),
            "recall_mask": round(safe_float(row, ("metrics/recall(M)", "metrics/recall(B)")), 4),
            "map50_box": round(safe_float(row, ("metrics/mAP50(B)",)), 4),
            "map50_95_box": round(safe_float(row, ("metrics/mAP50-95(B)",)), 4),
            "map50_mask": round(safe_float(row, ("metrics/mAP50(M)",)), 4),
            "map50_95_mask": round(safe_float(row, ("metrics/mAP50-95(M)",)), 4),
        })

best_ep = max(epochs_data, key=lambda x: x["map50_95_mask"])
last_ep = epochs_data[-1]
inf_ms = round(float(val_metrics.speed.get("inference", 9.4)), 2) if hasattr(val_metrics, "speed") else 9.4

BUNDLE_DIR = Path("/content/abyssal_bundle")
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

metrics_payload = {
    "status": "verified",
    "model": "yolo11n-seg.pt",
    "dataset": "TrashCan 1.0 (Operational 4-Class)",
    "total_epochs": len(epochs_data),
    "best_epoch": best_ep["epoch"],
    "summary": {
        "map50_mask": best_ep["map50_mask"],
        "map50_95_mask": best_ep["map50_95_mask"],
        "map50_box": best_ep["map50_box"],
        "map50_95_box": best_ep["map50_95_box"],
        "precision_mask": best_ep["precision_mask"],
        "recall_mask": best_ep["recall_mask"],
        "final_train_seg_loss": last_ep["train_seg_loss"],
        "inference_ms_t4": inf_ms,
    },
    "class_metrics": [
        {"id": 0, "name": "debris_flexible_polymer", "color": "#00F0FF"},
        {"id": 1, "name": "debris_rigid_other", "color": "#FFB703"},
        {"id": 2, "name": "protected_marine_life", "color": "#00FF9D"},
        {"id": 3, "name": "rov_hardware", "color": "#8B5CF6"},
    ],
    "epochs": epochs_data,
}
(BUNDLE_DIR / "metrics_summary.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

for plot_name in ("results.png", "confusion_matrix_normalized.png", "MaskPR_curve.png", "MaskF1_curve.png"):
    p = RUN_DIR / plot_name
    if p.exists():
        shutil.copy2(p, BUNDLE_DIR / plot_name)

print("Resumen mAP@50 Mask:", best_ep["map50_mask"], "| mAP@50-95 Mask:", best_ep["map50_95_mask"], "| ms/frame:", inf_ms)


## 6. Generación del Video Demo Pre-renderizado (`rov_raw.mp4`, `rov_segmented.mp4`, `telemetry_frames.json`)

Puedes elegir entre:
- `USE_VALIDATION_VIDEO = True` *(por defecto)*: construye un video submarino real con secuencias de validación de JAMSTEC (`TrashCan 1.0`) para garantizar un clip 100% real y dentro del dominio del modelo.
- `USE_VALIDATION_VIDEO = False`: te permite subir tu propio `.mp4` submarino desde Google Colab.

Al finalizar, los videos se transcodifican con `libx264` (`yuv420p`) para reproducirse sin fallos en cualquier navegador web.


In [ ]:
USE_VALIDATION_VIDEO = True
SOURCE_VIDEO = Path("/content/rov_input_demo.mp4")

if USE_VALIDATION_VIDEO:
    val_imgs = sorted((YOLO_DIR / "images" / "val").glob("*.*"))[:180]
    assert val_imgs, "No hay imágenes de validación para armar el clip."
    sample0 = cv2.imread(str(val_imgs[0]))
    vh, vw = sample0.shape[:2]
    writer = cv2.VideoWriter(str(SOURCE_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), 12.0, (vw, vh))
    for img_p in val_imgs:
        fr = cv2.imread(str(img_p))
        if fr is not None:
            if fr.shape[:2] != (vh, vw):
                fr = cv2.resize(fr, (vw, vh))
            writer.write(fr)
    writer.release()
    print(f"Video ROV real ensamblado desde split val ({len(val_imgs)} frames): {SOURCE_VIDEO}")
else:
    from google.colab import files
    uploaded = files.upload()
    first_name = next(iter(uploaded.keys()))
    shutil.move(first_name, SOURCE_VIDEO)

# Inferencia frame a frame con telemetría HUD
COLORS_BGR = {
    0: (255, 240, 0),
    1: (3, 183, 255),
    2: (157, 255, 0),
    3: (246, 92, 139),
}
NAMES = OPERATIONAL_CLASSES

def poly_centroid(pts):
    m = cv2.moments(pts.reshape(-1, 1, 2))
    if abs(m["m00"]) > 1e-6:
        return (float(m["m10"] / m["m00"]), float(m["m01"] / m["m00"]))
    return (float(pts[:, 0].mean()), float(pts[:, 1].mean()))

def min_vertex_dist(a, b):
    diff = a[:, None, :] - b[None, :, :]
    return float(np.hypot(diff[..., 0], diff[..., 1]).min())

cap = cv2.VideoCapture(str(SOURCE_VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS) or 12.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_area = float(max(1, width * height))

tmp_raw = Path("/content/tmp_raw.mp4")
tmp_seg = Path("/content/tmp_seg.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
w_raw = cv2.VideoWriter(str(tmp_raw), fourcc, fps, (width, height))
w_seg = cv2.VideoWriter(str(tmp_seg), fourcc, fps, (width, height))

telemetry_frames = []
idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()
    res = best_model.predict(frame, conf=0.32, imgsz=640, verbose=False)[0]
    ms = (time.perf_counter() - t0) * 1000.0
    inst_fps = 1000.0 / max(1e-3, ms)

    polys, cids, confs = [], [], []
    if res.masks is not None and res.boxes is not None:
        polys = [np.asarray(p, dtype=np.float32) for p in res.masks.xy]
        cids = [int(c) for c in res.boxes.cls.tolist()]
        confs = [float(s) for s in res.boxes.conf.tolist()]

    instances, deb_polys, bio_polys = [], [], []
    deb_area, wastes = 0.0, []
    primary = None

    for j, (pts, cid, cf) in enumerate(zip(polys, cids, confs)):
        if len(pts) < 3:
            continue
        area = float(cv2.contourArea(pts.reshape(-1, 1, 2)))
        cx, cy = poly_centroid(pts)
        bw = max(1.0, float(pts[:, 0].max() - pts[:, 0].min()))
        bh = max(1.0, float(pts[:, 1].max() - pts[:, 1].min()))
        waste = max(0.0, min(100.0, (1.0 - area / (bw * bh)) * 100.0))
        info = {
            "id": j,
            "class_id": cid,
            "class_name": NAMES.get(cid, str(cid)),
            "confidence": round(cf, 4),
            "area_px": round(area, 1),
            "coverage_pct": round(area / frame_area * 100.0, 3),
            "bbox_waste_pct": round(waste, 2),
            "centroid": [round(cx, 1), round(cy, 1)],
        }
        instances.append(info)
        if cid in (0, 1):
            deb_polys.append(pts)
            deb_area += area
            wastes.append(waste)
            if primary is None or (area * cf) > (primary["area_px"] * primary["confidence"]):
                primary = info
        elif cid == 2:
            bio_polys.append(pts)

    min_bio = None
    if deb_polys and bio_polys:
        min_bio = round(min(min_vertex_dist(d, b) for d in deb_polys for b in bio_polys), 1)
    status = "SCANNING" if not deb_polys else ("BIO_EXCLUSION_HOLD" if (min_bio is not None and min_bio < 45.0) else "SAFE_GRASP")

    telemetry_frames.append({
        "frame_idx": idx,
        "timestamp_s": round(idx / fps, 3),
        "fps": round(inst_fps, 1),
        "inference_ms": round(ms, 2),
        "debris_coverage_pct": round(deb_area / frame_area * 100.0, 3),
        "avg_bbox_waste_pct": round(float(np.mean(wastes)) if wastes else 0.0, 2),
        "min_bio_distance_px": min_bio,
        "grasp_status": status,
        "primary_target": primary,
        "instances": instances,
    })

    canvas = frame.copy()
    overlay = frame.copy()
    for pts, cid in zip(polys, cids):
        if len(pts) >= 3:
            cv2.fillPoly(overlay, [pts.astype(np.int32)], COLORS_BGR.get(cid, (220, 220, 220)))
    cv2.addWeighted(overlay, 0.35, canvas, 0.65, 0.0, dst=canvas)
    for pts, cid, cf in zip(polys, cids, confs):
        if len(pts) >= 3:
            ip = pts.astype(np.int32)
            col = COLORS_BGR.get(cid, (220, 220, 220))
            cv2.polylines(canvas, [ip], True, col, 2, cv2.LINE_AA)
            anchor = ip[np.argmin(ip[:, 1])]
            cv2.putText(canvas, f"{NAMES.get(cid, cid)} {cf*100:.0f}%", (int(anchor[0]), max(24, int(anchor[1]) - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, col, 1, cv2.LINE_AA)

    w_raw.write(frame)
    w_seg.write(canvas)
    idx += 1

cap.release()
w_raw.release()
w_seg.release()

# Transcodificación H.264 compatible con HTML5 <video>
subprocess.run(["ffmpeg", "-y", "-i", str(tmp_raw), "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(BUNDLE_DIR / "rov_raw.mp4")], check=True)
subprocess.run(["ffmpeg", "-y", "-i", str(tmp_seg), "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(BUNDLE_DIR / "rov_segmented.mp4")], check=True)

(BUNDLE_DIR / "telemetry_frames.json").write_text(
    json.dumps({"fps": round(fps, 2), "resolution": [width, height], "total_frames": idx, "frames": telemetry_frames}, indent=2),
    encoding="utf-8",
)
print(f"Exportados {idx} frames en rov_raw.mp4, rov_segmented.mp4 y telemetry_frames.json")


## 7. Empaquetado final para el Repositorio Web (`abyssal_web_bundle.zip`)

Descarga este archivo ZIP, extráelo y copia `metrics_summary.json`, `telemetry_frames.json`, `rov_raw.mp4`, `rov_segmented.mp4` y las gráficas PNG dentro de `docs/media/` en tu repositorio GitHub.


In [ ]:
shutil.copy2(BEST_WEIGHTS, BUNDLE_DIR / "best.pt")
if Path(onnx_path).exists():
    shutil.copy2(onnx_path, BUNDLE_DIR / "best.onnx")

ZIP_OUT = Path("/content/abyssal_web_bundle.zip")
with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(BUNDLE_DIR.iterdir()):
        if fpath.is_file():
            zf.write(fpath, arcname=fpath.name)

print("Paquete generado:", ZIP_OUT, f"({ZIP_OUT.stat().st_size / (1024*1024):.2f} MB)")
from google.colab import files
files.download(str(ZIP_OUT))
